# Master Extraction Notebook: NSSF Tier II Excel to TXT
One notebook for every file. **For each new Excel file, you change only the settings cell (cell 1).** Everything else stays the same, and you can use **Run All**.

All the special cases from the earlier files are now settings, not separate scripts:

| Problem we met | Setting that handles it |
|---|---|
| Different file, different headings (`e`, `Payroll Area`, `FACTORY CODE (Station)`, `Factory code`) | `columns` |
| A sheet that must be left out (`MASTER DATA`, or Kapkatet's last sheet) | `skip_sheets` |
| A heading cell that is empty (`Unnamed: 15` or `Unnamed: 16`) | `rename_headings` |
| Headings that start in a lower row (`MAY 2026`, Excel row 3) | `header_rows` |
| A sheet with no headings at all (`SEPT.`) | `no_heading_sheets` |
| Empty cells written as `Null` (the Kapkatet script) | `fill_empty` |
| A grand total line at the end of the file | `add_grand_total` |

What the code does for every sheet: keeps your columns, removes rows that have no `Pers NO` (this removes Excel's own total row), turns the amounts into real numbers, removes hidden spaces from the voluntary column, adds a total row for the month, and finally writes the pipe-delimited txt file. The last cell checks the txt file against the Excel sheets.

**Before the first run of a new file:** `.xls` files need `%pip install xlrd` in its own cell, then restart the kernel.

## Cell 1: Settings (the only cell you edit)
- Headings are **trimmed** after loading, so write them in `columns` **without** leading or trailing spaces (`"Payroll Area"`, not `"Payroll Area "`).
- Write sheet names **exactly** as they appear (copy them from the printout in cell 2).
- Use the right-click **Copy Path** option in VS Code for `path`.

In [ ]:
path = r"C:\Users\USER\...\YOUR FILE NAME.xlsx"
output_file = "YourFileName.txt"

# The headings to keep, in the order they must appear in the txt file
columns = ["Payroll Area", "Factory code", "Pers NO", "Voluntary Amount", "Tier2 Cont", "Payroll  Month"]

id_col = "Pers NO"                  # a row without a value here is not an employee
amount_col = "Tier2 Cont"           # the column that is added up
voluntary_col = "Voluntary Amount"  # cleaned of hidden spaces (use None if you have no such column)

skip_sheets = []                    # for example ["MASTER DATA"]
rename_headings = {}                # for example {"Unnamed: 15": "Payroll  Month"}
header_rows = {}                    # for example {"MAY 2026": 2}   (headings in Excel row 3 -> 2)
no_heading_sheets = {}              # for example {"SEPT.": ["Payroll Area", "Factory code", "Pers NO", "Voluntary Amount", "Tier2 Cont", "Payroll  Month"]}

fill_empty = None                   # None leaves empty cells empty; use "Null" to write Null
add_grand_total = True              # False leaves out the final grand total line

## Cell 2: Load the file
This reads every sheet, re-reads the special ones you listed, trims spaces from the headings, and applies your renames.

In [ ]:
import os
import pandas as pd

all_sheets = pd.read_excel(path, sheet_name=None, dtype={id_col: str})

for name, header_row in header_rows.items():
    all_sheets[name] = pd.read_excel(path, sheet_name=name, header=header_row, dtype={id_col: str})

for name, names in no_heading_sheets.items():
    all_sheets[name] = pd.read_excel(path, sheet_name=name, header=None, names=names, dtype={id_col: str})

for name, df in all_sheets.items():
    df.columns = df.columns.astype(str).str.strip()
    all_sheets[name] = df.rename(columns=rename_headings)

print([repr(name) for name in all_sheets.keys()])

## Cell 3: Look at the headings of every sheet
One column per sheet, one row per heading position. Use it to fill in the settings: look for `Unnamed:` names, missing columns, and different spellings.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

pd.DataFrame({name: pd.Series(df.columns) for name, df in all_sheets.items()}).fillna("")

## Cell 4: Check that every sheet has the columns you asked for
This prevents the `KeyError` surprises. Fix the settings until it says everything is fine.

In [ ]:
problems = 0
for sheet_name, sheet_df in all_sheets.items():
    if sheet_name in skip_sheets:
        continue
    missing = [c for c in columns if c not in sheet_df.columns]
    repeated = [c for c in columns if (sheet_df.columns == c).sum() > 1]
    if missing:
        print(repr(sheet_name), "is missing:", missing)
        problems += 1
    if repeated:
        print(repr(sheet_name), "has a repeated heading:", repeated)
        problems += 1

print("Everything is fine" if problems == 0 else str(problems) + " problem(s) to fix in the settings")

## Cell 5: Extract and write the txt file
Read the printed line for each sheet: the number of employee rows and the month total. Compare one or two of them with Excel before you trust the file.

In [ ]:
def clean_amount(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r"[^\d.-]", "", regex=True),
        errors="coerce"
    ).fillna(0)

combined = pd.DataFrame()
month_totals = []

for sheet_name, sheet_df in all_sheets.items():
    if sheet_name in skip_sheets:
        continue

    missing = [c for c in columns if c not in sheet_df.columns]
    if missing:
        raise ValueError("Sheet " + repr(sheet_name) + " is missing " + str(missing))

    selected = sheet_df[columns].copy()

    has_id = selected[id_col].notna() & (selected[id_col].astype(str).str.strip() != "")
    selected = selected[has_id].copy()

    selected[amount_col] = clean_amount(selected[amount_col])

    if fill_empty is not None:
        selected = selected.fillna(fill_empty)

    if voluntary_col is not None and voluntary_col in columns:
        selected[voluntary_col] = (
            selected[voluntary_col].astype(str)
            .str.replace("\xa0", "", regex=False)
            .str.strip()
            .replace("nan", "")
        )

    month_total = selected[amount_col].sum()
    month_totals.append(month_total)
    print(sheet_name, "| rows:", len(selected), "| total:", month_total)

    total_row = pd.DataFrame([{**{c: "" for c in columns}, amount_col: month_total}])
    combined = pd.concat([combined, selected, total_row], ignore_index=True)

if add_grand_total:
    grand_total = sum(month_totals)
    print("Grand total:", grand_total)
    grand_total_row = pd.DataFrame([{**{c: "" for c in columns}, amount_col: grand_total}])
    combined = pd.concat([combined, grand_total_row], ignore_index=True)

combined.to_csv(output_file, sep="|", index=False, header=False)
print("Saved:", os.path.abspath(output_file))

## Cell 6: Check the txt file against the Excel sheets
This reads the txt file back and compares it with the sheets: the row counts, each month's total, the sheet's own total row (when it has one), and every column, row by row.

How to read the table:
- **Rows in Excel** and **Rows in txt** must be equal.
- **Totals match** must be `True`.
- **Matches sheet's own total** should be `True`, or `no total row`. A `False` means Excel's own total and the employee rows disagree. Look at that sheet in Excel (for example a total formula that starts one row too low).
- **Columns that differ** must say `none`.

It prints no individual values. The check uses the same rules as the extraction, so it cannot catch a wrong rule. Compare one month with Excel by hand once per file, and check that the comma in your amounts is a thousands separator (`1,234.50`).

In [ ]:
def tidy(s):
    s = s.where(s.notna(), "" if fill_empty is None else fill_empty).astype(str)
    return s.str.replace("\xa0", "", regex=False).str.strip()

def to_num(s):
    return pd.to_numeric(tidy(s).str.replace(r"[^\d.-]", "", regex=True), errors="coerce")

number_cols = [c for c in [amount_col, voluntary_col] if c is not None and c in columns]

out = pd.read_csv(output_file, sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
is_employee = out[id_col].str.strip() != ""
is_total = ~is_employee & (out[amount_col].str.strip() != "")
block = is_total.cumsum()
file_totals = to_num(out.loc[is_total, amount_col]).round(2).tolist()

sheets_used = [(n, d) for n, d in all_sheets.items() if n not in skip_sheets]
expected_totals = len(sheets_used) + (1 if add_grand_total else 0)
totals_ok = len(file_totals) == expected_totals
if not totals_ok:
    print("WARNING: expected", expected_totals, "total rows in the txt file, found", len(file_totals))

results = []
for i, (name, df) in enumerate(sheets_used):
    amt = clean_amount(df[amount_col])
    has_id = df[id_col].notna() & (df[id_col].astype(str).str.strip() != "")
    exp = df.loc[has_id, columns].reset_index(drop=True)
    got = out[is_employee & (block == i)][columns].reset_index(drop=True)

    employee_sum = round(amt[has_id].sum(), 2)
    own_total = round(amt[~has_id].sum(), 2)

    row = {"Sheet": name, "Rows in Excel": len(exp), "Rows in txt": len(got),
           "Employee sum": employee_sum,
           "Sheet's own total row": own_total if own_total != 0 else None}

    if totals_ok:
        row["Total in txt"] = file_totals[i]
        row["Totals match"] = abs(employee_sum - file_totals[i]) < 0.005
    else:
        row["Total in txt"] = None
        row["Totals match"] = None

    row["Matches sheet's own total"] = "no total row" if own_total == 0 else bool(abs(employee_sum - own_total) < 0.005)

    if len(exp) != len(got):
        row["Columns that differ"] = "row counts differ"
    else:
        bad = []
        for col in columns:
            if col in number_cols:
                a, b = to_num(exp[col]), to_num(got[col])
                same = (a.isna() & b.isna()) | ((a - b).abs() < 0.005)
            else:
                same = tidy(exp[col]) == tidy(got[col])
            if not same.all():
                bad.append(col + " (" + str(int((~same).sum())) + " rows)")
        row["Columns that differ"] = ", ".join(bad) if bad else "none"
    results.append(row)

compare = pd.DataFrame(results)

if add_grand_total and totals_ok:
    print("Grand total correct:", abs(file_totals[-1] - compare["Employee sum"].sum()) < 0.005)
print("Sheets identical to the txt:", int((compare["Columns that differ"] == "none").sum()), "of", len(compare))

compare